In [1]:
import numpy as np
import os

folder = r"D:\Sign Lang Landmarks"

for f in os.listdir(folder):
    full_path = os.path.join(folder, f)
    if f.endswith('.npy'):
        arr = np.load(full_path, allow_pickle=True)
        print(f"{f}: shape={arr.shape}, dtype={arr.dtype}")
    else:
        print(f"{f}: (not npy)")

processed: (not npy)


In [2]:
import os

folder = r"D:\Sign Lang Landmarks\processed"

# See what's inside
for item in os.listdir(folder):
    full_path = os.path.join(folder, item)
    if os.path.isdir(full_path):
        files = os.listdir(full_path)
        print(f"📁 {item}/  → {len(files)} files")
        # Show first 3 files
        for f in files[:3]:
            print(f"     {f}")
    else:
        print(f"📄 {item}")

📁 test/  → 52 files
     Ambulance
     Bad
     Bandage
📁 train/  → 52 files
     Ambulance
     Bad
     Bandage
📁 val/  → 52 files
     Ambulance
     Bad
     Bandage


In [3]:
import os

folder = r"D:\Sign Lang Landmarks\processed"

# Look inside train/Ambulance
sample_path = os.path.join(folder, "train", "Ambulance")
files = os.listdir(sample_path)
print(f"Files in train/Ambulance: {len(files)}")
print(f"First 5 files: {files[:5]}")
print()

# Check one file
first_file = os.path.join(sample_path, files[0])
size = os.path.getsize(first_file) / 1024
print(f"First file: {files[0]}")
print(f"Size: {size:.1f} KB")
print(f"Extension: {os.path.splitext(files[0])[1]}")

# Try loading it
import numpy as np
try:
    arr = np.load(first_file, allow_pickle=True)
    print(f"Shape: {arr.shape}")
    print(f"Dtype: {arr.dtype}")
except Exception as e:
    print(f"Error loading as npy: {e}")

Files in train/Ambulance: 203
First 5 files: ['aug_Ambulance_0.npy', 'aug_Ambulance_1.npy', 'aug_Ambulance_10.npy', 'aug_Ambulance_100.npy', 'aug_Ambulance_101.npy']

First file: aug_Ambulance_0.npy
Size: 78.6 KB
Extension: .npy
Shape: (69, 291)
Dtype: float32


In [6]:
import numpy as np
import os
import json
from tqdm import tqdm

# ── Config ───────────────────────────────────────────────────
DATA_PATH  = r"D:\Sign Lang Landmarks\processed"
FIXED_LEN  = 100   # fixed sequence length
FEAT_DIM   = 291   # features per frame

# Load class mapping
with open(r"E:\Sign_Train_First\class_mapping.json") as f:
    sign_to_idx = json.load(f)

NUM_CLASSES = len(sign_to_idx)
print(f"Classes: {NUM_CLASSES}")
print(f"Fixed length: {FIXED_LEN}")
print(f"Feature dim: {FEAT_DIM}")

def pad_or_truncate(seq, fixed_len):
    """
    Pad sequence with zeros if shorter than fixed_len
    Truncate if longer than fixed_len
    """
    seq_len = seq.shape[0]

    if seq_len >= fixed_len:
        # Truncate — take middle frames
        start = (seq_len - fixed_len) // 2
        return seq[start:start + fixed_len]
    else:
        # Pad with zeros at end
        pad = np.zeros((fixed_len - seq_len, seq.shape[1]), dtype=np.float32)
        return np.vstack([seq, pad])

def load_split(split_name):
    """Load all samples from one split (train/val/test)"""
    split_path = os.path.join(DATA_PATH, split_name)
    X = []
    y = []

    classes = sorted(os.listdir(split_path))

    for class_name in classes:
        if class_name not in sign_to_idx:
            print(f"  ⚠️ Skipping unknown class: {class_name}")
            continue

        label     = sign_to_idx[class_name]
        class_path = os.path.join(split_path, class_name)
        files     = os.listdir(class_path)

        for fname in files:
            fpath = os.path.join(class_path, fname)
            try:
                seq = np.load(fpath)              # (variable_len, 291)
                seq = pad_or_truncate(seq, FIXED_LEN)  # (100, 291)
                X.append(seq)
                y.append(label)
            except Exception as e:
                print(f"  ❌ Error loading {fname}: {e}")

        print(f"  ✅ {class_name:20s}: {len(files)} samples")

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

# Load all splits
print("\n📂 Loading TRAIN...")
X_train, y_train = load_split("train")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

print("\n📂 Loading VAL...")
X_val, y_val = load_split("val")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")

print("\n📂 Loading TEST...")
X_test, y_test = load_split("test")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Classes: 52
Fixed length: 100
Feature dim: 291

📂 Loading TRAIN...
  ✅ Ambulance           : 203 samples
  ✅ Bad                 : 203 samples
  ✅ Bandage             : 203 samples
  ✅ Book                : 203 samples
  ✅ Come                : 203 samples
  ✅ Cough               : 203 samples
  ✅ Doctor              : 203 samples
  ✅ Eat                 : 203 samples
  ✅ Elder Brother       : 203 samples
  ✅ Elder Sister        : 203 samples
  ✅ Father              : 203 samples
  ✅ Fever               : 203 samples
  ✅ Friday              : 203 samples
  ✅ Go                  : 203 samples
  ✅ Good                : 203 samples
  ✅ Headache            : 203 samples
  ✅ Help                : 203 samples
  ✅ Here                : 203 samples
  ✅ Home                : 203 samples
  ✅ Hospital            : 203 samples
  ✅ I or Me             : 203 samples
  ✅ Injection           : 203 samples
  ✅ Left                : 203 samples
  ✅ Medicine            : 203 samples
  ✅ Monday           

In [5]:
import os

# Search for class_mapping.json
for root, dirs, files in os.walk(r"D:\Sign Lang Landmarks"):
    for f in files:
        print(os.path.join(root, f))

D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_0.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_1.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_10.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_11.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_12.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_13.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_14.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_15.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_16.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_17.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_18.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_19.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_2.npy
D:\Sign Lang Landmarks\processed\test\Ambulance\aug_Ambulance_20.npy
D:\Sign Lang Landmarks\processed\test